# PSF-Zero: Analytic KAK Decomposition (rewritten 2026-09-24)

**This notebook replaces an earlier version of this file that described an
iterative, gradient-based optimizer (`PSFHybridSynthesizer`, an Adam loop
over `PSFHyper`).** That description does not match what PSF-Zero actually
is or does, and the module it imported (`psf_synthesis.py`) is not part of
this project's own current, verified code.

**What PSF-Zero actually does** (see the project README's own opening
example, which this notebook follows): it replaces heuristic two-qubit
unitary synthesis with an **exact, closed-form Cartan (KAK) decomposition**,
computed by a small Rust core. There is no optimizer, no loss function, no
iteration count, and no random seed to control for -- the same input unitary
always produces the same output circuit, because the decomposition is
analytic rather than search-based.

This notebook:
1. builds a random 2-qubit unitary,
2. compiles it with `psf_compile.compile()` -- the project's own real,
   public entry point,
3. checks the result against Qiskit's own `Operator` equivalence (the same
   check this project's benchmarks use throughout),
4. repeats this over several random unitaries and reports the worst-case
   infidelity -- the same methodology as
   `benchmarks/verify_core_infidelity.py`, on a small scale suitable for a
   notebook.


In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import Operator, random_unitary

from psf_compile import compile as psf_compile

## 1. A single random 2-qubit unitary

`psf_compile.compile()` takes a `QuantumCircuit` and looks for two-qubit
blocks it can replace; here the circuit is nothing but one such block, so
the whole circuit gets synthesized.

In [ ]:
rng_seed = 42
U_target = random_unitary(4, seed=rng_seed)

qc = QuantumCircuit(2)
qc.append(UnitaryGate(U_target), [0, 1])

qc_synth = psf_compile(qc, verify=False)
print(qc_synth.draw())

## 2. Correctness: exact `Operator` equivalence

Qiskit's own `Operator` equivalence, up to global phase, on the synthesized
circuit versus the original unitary -- the same check
`verify_core_infidelity.py` uses, not a separately-invented one.

In [ ]:
equiv = Operator(qc_synth).equiv(Operator(U_target))
print(f"Operator.equiv(): {equiv}")
assert equiv, "synthesized circuit does not match the target unitary"

# A numeric infidelity figure as well, for comparison against the README's
# own reported numbers (worst case 1.11e-15 on 500 Haar-random samples).
def gate_infidelity(target, candidate):
    d = target.dim[0]
    tr = np.trace(target.data.conj().T @ candidate.data)
    return 1.0 - (abs(tr) ** 2 + d) / (d * (d + 1))

infid = gate_infidelity(Operator(U_target), Operator(qc_synth))
print(f"gate infidelity: {infid:.2e}")

## 3. Repeated over several random unitaries

No optimizer, no convergence curve -- each call is a single, deterministic,
closed-form decomposition. What varies across samples is only the target
unitary; the worst-case infidelity across the batch is what
`verify_core_infidelity.py` reports as the project's own accuracy figure.

In [ ]:
n_samples = 50
worst_infid = 0.0
gate_counts = []
infidelities = []  # kept for the plot below, not just the running worst-case

for seed in range(n_samples):
    U = random_unitary(4, seed=seed)
    qc = QuantumCircuit(2)
    qc.append(UnitaryGate(U), [0, 1])
    qc_out = psf_compile(qc, verify=False)

    infid = gate_infidelity(Operator(U), Operator(qc_out))
    worst_infid = max(worst_infid, infid)
    gate_counts.append(qc_out.size())
    infidelities.append(infid)

print(f"worst infidelity over {n_samples} samples: {worst_infid:.2e}")
print(f"gate count per synthesized block: min={min(gate_counts)}, "
      f"max={max(gate_counts)}, all equal={len(set(gate_counts)) == 1}")

**Reading the result**: the gate count should be identical across every
sample -- the decomposition is analytic, so the *shape* of the output
circuit does not depend on which unitary was supplied, only the angles
inside it do. If `all equal` above is `False`, that is worth investigating
before trusting anything else in this notebook, since it would mean this
run's own installed `psf_compile.py` behaves differently from what the
project's README documents.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(infidelities, bins=15, color="#00ffcc")
ax.set_xlabel("Gate infidelity (per random 2-qubit unitary)")
ax.set_ylabel("Count")
ax.set_title("PSF-Zero: infidelity across random Haar unitaries")
plt.tight_layout()
plt.show()